In [1]:
import simpy
import random
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
from tqdm import tqdm
from gurobipy import Model, GRB, quicksum
from collections import defaultdict

In [2]:
class CLASS:

    def __init__(self, service_rate, dropout_rate, dropout_cost, anandonment_rate_without_AI,
                anandonment_cost_without_AI, if_use_AI_or_not, supervision_cost,
                negative_anandonment_rate, positive_anandonment_rate,
                negative_anandonment_cost, holding_cost_without_AI, holding_cost_with_AI, benefit, arrival_rate,name,
                fixed_overhead_cost):
        
        self.service_rate = service_rate
        self.dropout_rate = dropout_rate
        self.dropout_cost = dropout_cost
        self.anandonment_rate_without_AI = anandonment_rate_without_AI
        self.anandonment_cost_without_AI = anandonment_cost_without_AI
        self.if_use_AI_or_not = if_use_AI_or_not
        self.supervision_cost = supervision_cost
        self.negative_anandonment_rate = negative_anandonment_rate
        self.positive_anandonment_rate = positive_anandonment_rate
        self.negative_anandonment_cost = negative_anandonment_cost
        self.holding_cost_without_AI = holding_cost_without_AI
        self.holding_cost_with_AI = holding_cost_with_AI
        self.benefit = benefit
        self.arrival_rate = arrival_rate
        self.fixed_overhead_cost = fixed_overhead_cost 
        self.name = name
        
        r = benefit * service_rate - dropout_rate * dropout_cost
        c = holding_cost_without_AI + anandonment_rate_without_AI * anandonment_cost_without_AI
        theta_w = negative_anandonment_rate + positive_anandonment_rate
        c_w = holding_cost_with_AI + negative_anandonment_cost * negative_anandonment_rate + supervision_cost - positive_anandonment_rate * benefit
        
        P = (r * anandonment_rate_without_AI + c * (service_rate + dropout_rate)) / anandonment_rate_without_AI
        P_w = (r * theta_w + c_w * (service_rate + dropout_rate)) / theta_w

        self.r = r
        self.c = c
        self.c_w = c_w
        self.P = P
        self.P_w = P_w

        self.z_upper = arrival_rate / (service_rate + dropout_rate)

        self.in_system = 0
        self.in_service = 0

In [3]:
def solve_ai_waitlist_milp(classes, N):

    m = Model("AI_waitlist_MILP")
    
    m.Params.OutputFlag = 0

    I = range(len(classes))

    z = m.addVars(I, lb=0.0, name="z")               
    w = m.addVars(I, vtype=GRB.BINARY, name="w")     
    y = m.addVars(I, lb=0.0, name="y")             

    params = {}
    for i, cls in enumerate(classes):
        mu = cls.service_rate
        gamma = cls.dropout_rate
        lam = cls.arrival_rate

        b = cls.benefit
        h = cls.holding_cost_without_AI
        h_w = cls.holding_cost_with_AI

        theta_n = cls.anandonment_rate_without_AI
        alpha_n = cls.anandonment_cost_without_AI

        theta_wp = cls.positive_anandonment_rate
        theta_wn = cls.negative_anandonment_rate
        alpha_wn = cls.negative_anandonment_cost

        a = cls.supervision_cost
        f_bar = cls.fixed_overhead_cost

        r = b * mu - cls.dropout_cost * gamma
        c = h + alpha_n * theta_n
        theta_w = theta_wp + theta_wn
        c_w = h_w + alpha_wn * theta_wn + a - theta_wp * b

        P = (r * theta_n + c * (mu + gamma)) / theta_n
        P_w = (r * theta_w + c_w * (mu + gamma)) / theta_w

        z_max = lam / (mu + gamma)

        A = c / theta_n
        B = c_w / theta_w

        params[i] = dict(
            mu=mu, gamma=gamma, lam=lam,
            r=r, c=c, c_w=c_w,
            theta_n=theta_n, theta_w=theta_w,
            P=P, P_w=P_w,
            z_max=z_max,
            A=A, B=B,
            f_bar=f_bar,
        )

    for i in I:
        m.addConstr(z[i] <= params[i]["z_max"], name=f"stab[{i}]")

    m.addConstr(quicksum(z[i] for i in I) <= N, name="capacity")

    for i in I:
        z_max = params[i]["z_max"]
        m.addConstr(y[i] <= z[i], name=f"mc1[{i}]")
        m.addConstr(y[i] <= z_max * w[i], name=f"mc2[{i}]")
        m.addConstr(y[i] >= z[i] - z_max * (1 - w[i]), name=f"mc3[{i}]")

    obj = 0
    for i in I:
        P = params[i]["P"]
        P_w = params[i]["P_w"]
        lam = params[i]["lam"]
        A = params[i]["A"]
        B = params[i]["B"]
        f_bar = params[i]["f_bar"]

        obj += P * z[i]
        obj += (P_w - P) * y[i]
        obj += -lam * (A + (B - A) * w[i])
        obj += -f_bar * w[i]

    m.setObjective(obj, GRB.MAXIMIZE)
    m.optimize()
    
    return m.ObjVal

In [4]:
def jitter(x, pct=0.10):
    """Return a random number within ±pct of x (uniform)."""
    return x * random.uniform(1 - pct, 1 + pct)

In [5]:
def one_step_reward(CLASS_A, CLASS_B, xA, xB, zA, zB):

    reward_rate = (
        min(xA, zA) * CLASS_A.r
        - max(0, xA - zA) * (CLASS_A.c_w * CLASS_A.if_use_AI_or_not + CLASS_A.c * (1 - CLASS_A.if_use_AI_or_not))
        + min(xB, zB) * CLASS_B.r
        - max(0, xB - zB) * (CLASS_B.c_w * CLASS_B.if_use_AI_or_not + CLASS_B.c * (1 - CLASS_B.if_use_AI_or_not))
        - CLASS_A.fixed_overhead_cost * CLASS_A.if_use_AI_or_not
        - CLASS_B.fixed_overhead_cost * CLASS_B.if_use_AI_or_not
    )
    return reward_rate

def compute_uniformization_rate(CLASS_A, CLASS_B, xA_max, xB_max, N):
   
    thetaA_max = max(
        CLASS_A.anandonment_rate_without_AI,
        CLASS_A.negative_anandonment_rate + CLASS_A.positive_anandonment_rate,
    )
    thetaB_max = max(
        CLASS_B.anandonment_rate_without_AI,
        CLASS_B.negative_anandonment_rate + CLASS_B.positive_anandonment_rate,
    )

    busyA_max = N
    busyB_max = N
    waitA_max = xA_max
    waitB_max = xB_max

    total_rate_max = (
        CLASS_A.arrival_rate
        + CLASS_B.arrival_rate
        + (CLASS_A.service_rate + CLASS_A.dropout_rate) * busyA_max
        + thetaA_max * waitA_max
        + (CLASS_B.service_rate + CLASS_B.dropout_rate) * busyB_max
        + thetaB_max * waitB_max
    )

    Lambda = total_rate_max #* 1.05
    return Lambda


def compute_transition_probs_and_reward(
    CLASS_A,
    CLASS_B,
    xA,
    xB,
    zA,
    zB,
    MP,
    Lambda,
    xA_max,
    xB_max,
):
   

  
    busyA = min(xA, zA)
    busyB = min(xB, zB)
    waitA = max(0, xA - busyA)
    waitB = max(0, xB - busyB)

   
    rate_dep_A = (CLASS_A.service_rate + CLASS_A.dropout_rate) * busyA
    rate_dep_B = (CLASS_B.service_rate + CLASS_B.dropout_rate) * busyB


    rate_aband_A = (
        (1 - CLASS_A.if_use_AI_or_not) * CLASS_A.anandonment_rate_without_AI
        + CLASS_A.if_use_AI_or_not
        * (CLASS_A.negative_anandonment_rate + CLASS_A.positive_anandonment_rate)
    ) * waitA

    rate_aband_B = (
        (1 - CLASS_B.if_use_AI_or_not) * CLASS_B.anandonment_rate_without_AI
        + CLASS_B.if_use_AI_or_not
        * (CLASS_B.negative_anandonment_rate + CLASS_B.positive_anandonment_rate)
    ) * waitB

    total_rate = (
        CLASS_A.arrival_rate
        + CLASS_B.arrival_rate
        + rate_dep_A
        + rate_aband_A
        + rate_dep_B
        + rate_aband_B
    )

    if total_rate > Lambda + 1e-9:
        raise ValueError(
            f"Lambda too small! total_rate={total_rate}, Lambda={Lambda}"
        )


    reward_rate = one_step_reward(CLASS_A, CLASS_B, xA, xB, zA, zB)

    step_reward = reward_rate / Lambda

    s = (xA, xB)

    def add_transition(nxA, nxB, prob):
       
        nxA_clamped = min(max(nxA, 0), xA_max)
        nxB_clamped = min(max(nxB, 0), xB_max)
        key = (s[0], s[1], nxA_clamped, nxB_clamped, zA, zB)

        if key in MP:
            old_prob, old_r = MP[key]
            if abs(old_r - step_reward) > 1e-10:
                raise ValueError(
                    f"Inconsistent reward for same (s,a,sp): old={old_r}, new={step_reward}"
                )
            MP[key] = (old_prob + prob, step_reward)
        else:
            MP[key] = (prob, step_reward)

    # A 
    add_transition(xA + 1, xB, CLASS_A.arrival_rate / Lambda)

    # B 
    add_transition(xA, xB + 1, CLASS_B.arrival_rate / Lambda)

    # A 
    if xA - 1 >= 0:
        add_transition(xA - 1, xB, (rate_dep_A + rate_aband_A) / Lambda)

    # B 
    if xB - 1 >= 0:
        add_transition(xA, xB - 1, (rate_dep_B + rate_aband_B) / Lambda)

   
    stay_prob = 1.0 - total_rate / Lambda
    if stay_prob < -1e-12:
        raise ValueError("Lambda too small, negative stay_prob")
    stay_prob = max(0.0, stay_prob)

    add_transition(xA, xB, stay_prob)

    return MP


def solve_MDP_from_sadata(
    sa_data,
    actions_by_state,
    states,
    state_index,
    epsilon=1e-8,
    max_iter=1000,
    verbose=True,
):
    

    num_states = len(states)

    h = np.zeros(num_states)
    s0 = states[0]
    i0 = state_index[s0]

    g_est = 0.0

    for it in range(max_iter):
        h_old = h.copy()
        v_new = np.zeros(num_states)

        for s in states:
            i = state_index[s]

            if s not in actions_by_state:
               
                v_new[i] = h_old[i]
                continue

            best_val = -1e30

            # Bellman : max_a [ r(s,a) + Σ p(s'|s,a) h_old(s') ]
            for a in actions_by_state[s]:
                data = sa_data[(s, a)]
                r_sa = data["reward"]  # per-step reward
                q_val = r_sa

                for (sp, prob) in data["trans"]:
                    j = state_index[sp]
                    q_val += prob * h_old[j]

                if q_val > best_val:
                    best_val = q_val

            v_new[i] = best_val

        g_est = v_new[i0]
        h = v_new - g_est

        diff = np.max(np.abs(h - h_old))

        if diff < epsilon:
            break

    g_tilde = g_est

    return g_tilde

In [6]:
def MDP_solution(CLASS_A, CLASS_B, xA_max, xB_max, N):

    Lambda = compute_uniformization_rate(CLASS_A, CLASS_B, xA_max, xB_max, N)
    
    MP = {}

    for xA in range(0, xA_max + 1):
        for xB in range(0, xB_max + 1):
            for zA in range(0, N + 1):
                for zB in range(0, N + 1 - zA):
                    compute_transition_probs_and_reward(
                        CLASS_A, CLASS_B,
                        xA,
                        xB,
                        zA,
                        zB,
                        MP,
                        Lambda,
                        xA_max, xB_max)

    states = [
        (xA, xB)
        for xA in range(xA_max + 1)
        for xB in range(xB_max + 1)
    ]
    state_index = {s: i for i, s in enumerate(states)}

    sa_data = defaultdict(lambda: {"reward": None, "trans": []})
    actions_by_state = defaultdict(list)

    for (xA, xB, nxA, nxB, zA, zB), (p, r_step) in MP.items():
        
        s = (xA, xB)
        sp = (nxA, nxB)
        a = (zA, zB)

        if (s not in state_index) or (sp not in state_index):
            continue

        data = sa_data[(s, a)]

        if data["reward"] is None:
            data["reward"] = float(r_step)
        else:
            if abs(data["reward"] - float(r_step)) > 1e-8:
                print(
                    "Warning: (state, action) has different reward",
                    "state=", s,
                    "action=", a,
                    "old_r=", data["reward"],
                    "new_r=", r_step,
                )

        data["trans"].append((sp, float(p)))

        if a not in actions_by_state[s]:
            actions_by_state[s].append(a)

    for (s, a), data in sa_data.items():
        probs = [p for (_, p) in data["trans"]]
        total_p = sum(probs)

    g_tilde = solve_MDP_from_sadata(
            sa_data,
            actions_by_state,
            states,
            state_index,
            epsilon=1e-6,
            max_iter=2000,
            verbose=True,
        )

    g_ct = Lambda * g_tilde
    
    return g_ct

In [7]:
# VHA case study calibrated parameters (weekly)
# Source: Section 6.1 in the provided paper PDF.

# Common parameters
N = 4

beta = 0.82                 # show-up probability
dropout_rate = 0.0067       # gamma per week
abandonment_rate_wo_ai = 0.04  # theta^n per week
positive_abandonment_rate = 0.06   # theta^{w,p} per week
negative_abandonment_rate = 0.0005 # theta^{w,n} per week
supervision_cost = 1        # a_i per patient per week
fixed_overhead_cost = 640   # fbar_i per class per week

# Benefits (b_i)
b_MDD = 12000
b_AD  = 9000

# Costs derived from benefit
abandonment_cost_wo_ai_MDD = 0.02 * b_MDD   # alpha^n
abandonment_cost_wo_ai_AD  = 0.02 * b_AD

negative_abandonment_cost_MDD = 0.01 * b_MDD  # alpha^{w,n}
negative_abandonment_cost_AD  = 0.01 * b_AD

dropout_cost_MDD = 0.01 * b_MDD  # h^d
dropout_cost_AD  = 0.01 * b_AD

# Holding costs (weekly)
holding_cost_wo_ai_MDD = 324
holding_cost_wo_ai_AD  = 196

holding_cost_w_ai_MDD = 32.4
holding_cost_w_ai_AD  = 19.6

# Service rates: mu = beta * mu^b
mu_b_MDD = 2.96
mu_b_AD  = 2.96

service_rate_MDD  = beta * mu_b_MDD   # 2.4272
service_rate_AD   = beta * mu_b_AD    # 2.4272

# Arrival rates (weekly)
arrival_rate_MDD  = 81/50*N
arrival_rate_AD   = 57/50*N

xA_max = 10*N
xB_max = 10*N

In [8]:
FLUID_RESULTS = []
MDP_RESULTS = []

for replication in tqdm(range(100)):

    random.seed(2026 + replication * 2026)
    
    CLASS_A = CLASS(
        service_rate=jitter(service_rate_MDD),
        dropout_rate=jitter(dropout_rate),
        dropout_cost=jitter(dropout_cost_MDD),
        anandonment_rate_without_AI=jitter(abandonment_rate_wo_ai),
        anandonment_cost_without_AI=jitter(abandonment_cost_wo_ai_MDD),
        if_use_AI_or_not=None,
        supervision_cost=jitter(supervision_cost),
        negative_anandonment_rate=jitter(negative_abandonment_rate),
        positive_anandonment_rate=jitter(positive_abandonment_rate),
        negative_anandonment_cost=jitter(negative_abandonment_cost_MDD),
        holding_cost_without_AI=jitter(holding_cost_wo_ai_MDD),
        holding_cost_with_AI=jitter(holding_cost_w_ai_MDD),
        benefit=jitter(b_MDD),
        arrival_rate=jitter(arrival_rate_MDD),
        fixed_overhead_cost=jitter(fixed_overhead_cost),
        name="MDD"
    )
    
    CLASS_B = CLASS(
        service_rate=jitter(service_rate_AD),
        dropout_rate=jitter(dropout_rate),
        dropout_cost=jitter(dropout_cost_AD),
        anandonment_rate_without_AI=jitter(abandonment_rate_wo_ai),
        anandonment_cost_without_AI=jitter(abandonment_cost_wo_ai_AD),
        if_use_AI_or_not=None,
        supervision_cost=jitter(supervision_cost),
        negative_anandonment_rate=jitter(negative_abandonment_rate),
        positive_anandonment_rate=jitter(positive_abandonment_rate),
        negative_anandonment_cost=jitter(negative_abandonment_cost_AD),
        holding_cost_without_AI=jitter(holding_cost_wo_ai_AD),
        holding_cost_with_AI=jitter(holding_cost_w_ai_AD),
        benefit=jitter(b_AD),
        arrival_rate=jitter(arrival_rate_AD),
        fixed_overhead_cost=jitter(fixed_overhead_cost),
        name="AD"
    )

    fluid_result = solve_ai_waitlist_milp(classes=[CLASS_A, CLASS_B], N=N)
    FLUID_RESULTS.append(fluid_result)

    mdp_result = []
    # ----------------------------------------------------

    CLASS_A.if_use_AI_or_not = 0
    CLASS_B.if_use_AI_or_not = 0
    mdp_result_a = MDP_solution(CLASS_A = CLASS_A, CLASS_B = CLASS_B, xA_max = xA_max, xB_max = xB_max, N = N)
    mdp_result.append(mdp_result_a)
    # ----------------------------------------------------

    CLASS_A.if_use_AI_or_not = 1
    CLASS_B.if_use_AI_or_not = 0
    mdp_result_b = MDP_solution(CLASS_A = CLASS_A, CLASS_B = CLASS_B, xA_max = xA_max, xB_max = xB_max, N = N)
    mdp_result.append(mdp_result_b)

    # ----------------------------------------------------

    CLASS_A.if_use_AI_or_not = 0
    CLASS_B.if_use_AI_or_not = 1
    mdp_result_c = MDP_solution(CLASS_A = CLASS_A, CLASS_B = CLASS_B, xA_max = xA_max, xB_max = xB_max, N = N)
    mdp_result.append(mdp_result_c)

    # ----------------------------------------------------

    CLASS_A.if_use_AI_or_not = 1
    CLASS_B.if_use_AI_or_not = 1
    mdp_result_d = MDP_solution(CLASS_A = CLASS_A, CLASS_B = CLASS_B, xA_max = xA_max, xB_max = xB_max, N = N)
    mdp_result.append(mdp_result_d)

    MDP_RESULTS.append(np.array(mdp_result).max())

  0%|                                                                                          | 0/100 [00:00<?, ?it/s]

Set parameter Username
Academic license - for non-commercial use only - expires 2027-03-08


100%|████████████████████████████████████████████████████████████████████████████| 100/100 [13:56:21<00:00, 501.81s/it]


In [9]:
(abs(np.array(FLUID_RESULTS)-np.array(MDP_RESULTS))/np.array(MDP_RESULTS)).mean()

0.008497557272026947

In [10]:
(abs(np.array(FLUID_RESULTS)-np.array(MDP_RESULTS))/np.array(MDP_RESULTS)).max()

0.01494999773831149

In [11]:
np.quantile((abs(np.array(FLUID_RESULTS)-np.array(MDP_RESULTS))/np.array(MDP_RESULTS)),0.025)

0.004143601030208061

In [12]:
np.quantile((abs(np.array(FLUID_RESULTS)-np.array(MDP_RESULTS))/np.array(MDP_RESULTS)),0.975)

0.014568618739148411